In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_DIR = Path.cwd().parent
DATA_FILE = PROJECT_DIR / "data" / "processed" / "lpmc.csv"

# Load dataset
df = pd.read_csv(DATA_FILE)

print("Observations:", len(df))
print("Variables:", len(df.columns))

display(df.head())


Observations: 81086
Variables: 32


,trip_id,household_id,person_n,trip_n,travel_mode,purpose,fueltype,faretype,bus_scale,survey_year,...,dur_pt_access,dur_pt_rail,dur_pt_bus,dur_pt_int,pt_interchanges,dur_driving,cost_transit,cost_driving_fuel,cost_driving_ccharge,driving_traffic_percent
0,0,0,0,0,4,3,1,1,1.0,1,...,0.134444,0.0,0.016667,0.0,0,0.052222,1.5,0.14,0.0,0.111702
1,1,0,0,1,4,3,1,1,1.0,1,...,0.109444,0.0,0.055556,0.0,0,0.059444,1.5,0.15,0.0,0.112150
2,2,0,0,2,4,3,1,1,1.0,1,...,0.203056,0.0,0.210278,0.0,0,0.236667,1.5,0.79,0.0,0.203052
3,3,0,0,3,4,3,1,1,1.0,1,...,0.205556,0.0,0.258611,0.0,0,0.233333,1.5,0.78,0.0,0.160714
4,4,0,1,2,4,3,1,4,1.0,1,...,0.203056,0.0,0.189444,0.0,0,0.229167,1.5,0.78,0.0,0.130909


In [2]:

# Travel time variables (hours)

df["time_walk"] = df["dur_walking"]
df["time_cycle"] = df["dur_cycling"]

df["time_pt"] = (
    df["dur_pt_access"]
    + df["dur_pt_rail"]
    + df["dur_pt_bus"]
    + df["dur_pt_int"]
)

df["time_car"] = df["dur_driving"]

# Travel costs (GBP)

df["cost_pt"] = df["cost_transit"]

df["cost_car"] = (
    df["cost_driving_fuel"]
    + df["cost_driving_ccharge"]
)

# Inspect prepared variables
model_cols = [
    "travel_mode",
    "time_walk",
    "time_cycle",
    "time_pt",
    "time_car",
    "cost_pt",
    "cost_car"
]

display(df[model_cols].head())


,travel_mode,time_walk,time_cycle,time_pt,time_car,cost_pt,cost_car
0,4,0.218056,0.092222,0.151111,0.052222,1.5,0.14
1,4,0.188889,0.061944,0.165000,0.059444,1.5,0.15
2,4,1.037778,0.355833,0.413333,0.236667,1.5,0.79
3,4,1.039444,0.384444,0.464167,0.233333,1.5,0.78
4,4,1.037778,0.355833,0.392500,0.229167,1.5,0.78


In [3]:

# Example parameter values (not estimated)

ASC_CYCLE = -1.0
ASC_PT = -0.5
ASC_CAR = -0.8

B_TIME = -2.0
B_COST = -0.3

# Select the first trip
trip = df.iloc[0]

V_walk = B_TIME * trip["time_walk"]

V_cycle = (
    ASC_CYCLE
    + B_TIME * trip["time_cycle"]
)

V_pt = (
    ASC_PT
    + B_TIME * trip["time_pt"]
    + B_COST * trip["cost_pt"]
)

V_car = (
    ASC_CAR
    + B_TIME * trip["time_car"]
    + B_COST * trip["cost_car"]
)

utilities = np.array([
    V_walk,
    V_cycle,
    V_pt,
    V_car
])

print("Utilities:")
print(utilities)


Utilities:
[-0.43611111 -1.18444444 -1.25222222 -0.94644444]


In [4]:

# Calculate MNL probabilities

exp_utilities = np.exp(
    utilities - np.max(utilities)
)

probabilities = (
    exp_utilities / exp_utilities.sum()
)

mode_names = [
    "Walk",
    "Cycle",
    "Public Transport",
    "Car"
]

results = pd.DataFrame({
    "Mode": mode_names,
    "Utility": utilities,
    "Probability": probabilities,
    "Probability (%)": probabilities * 100
})

display(results.round(4))

print(
    "Sum of probabilities:",
    probabilities.sum()
)

print(
    "Actual chosen mode:",
    mode_names[int(trip["travel_mode"]) - 1]
)


,Mode,Utility,Probability,Probability (%)
0,Walk,-0.4361,0.3975,39.7520
1,Cycle,-1.1844,0.1881,18.8088
2,Public Transport,-1.2522,0.1758,17.5763
3,Car,-0.9464,0.2386,23.8629


Sum of probabilities: 1.0
Actual chosen mode: Car


In [5]:

from scipy.optimize import minimize
from scipy.special import logsumexp

# Travel times: hours
T = df[
    ["time_walk", "time_cycle", "time_pt", "time_car"]
].to_numpy(dtype=float)

# Travel costs: GBP
C = np.column_stack([
    np.zeros(len(df)),  # Walk
    np.zeros(len(df)),  # Cycle
    df["cost_pt"],
    df["cost_car"]
])

# Observed travel choices: convert 1-4 to 0-3
y = df["travel_mode"].to_numpy(dtype=int) - 1

print("Travel time matrix:", T.shape)
print("Travel cost matrix:", C.shape)
print("Observed choices:", y.shape)


Travel time matrix: (81086, 4)
Travel cost matrix: (81086, 4)
Observed choices: (81086,)


In [6]:

def negative_log_likelihood(params, T, C, y):

    # Unpack parameters
    asc_cycle, asc_pt, asc_car, b_time, b_cost = params

    # Alternative-specific constants
    ASC = np.array([
        0.0,        # Walk (reference)
        asc_cycle,
        asc_pt,
        asc_car
    ])

    # Utilities for all observations
    V = ASC + b_time * T + b_cost * C

    # Log of the MNL denominator
    log_denominator = logsumexp(V, axis=1)

    # Utility of the actually chosen alternative
    chosen_utility = V[np.arange(len(y)), y]

    # Log-likelihood
    log_likelihood = np.sum(
        chosen_utility - log_denominator
    )

    return -log_likelihood


In [7]:

# Initial parameter values
initial_params = np.zeros(5)

# Maximum Likelihood Estimation
model = minimize(
    fun=negative_log_likelihood,
    x0=initial_params,
    args=(T, C, y),
    method="L-BFGS-B",
    options={"maxiter": 500}
)

parameter_names = [
    "ASC_Cycle",
    "ASC_PT",
    "ASC_Car",
    "B_Time",
    "B_Cost"
]

estimated_params = pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": model.x
})

print("Optimization successful:", model.success)
print("Message:", model.message)
print("Final Log-Likelihood:", -model.fun)

display(estimated_params.round(5))


Optimization successful: True
Message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Final Log-Likelihood: -74975.97607475249


,Parameter,Estimate
0,ASC_Cycle,-3.79813
1,ASC_PT,-0.50366
2,ASC_Car,-1.22338
3,B_Time,-5.31017
4,B_Cost,-0.17137


In [8]:

b_time = model.x[3]
b_cost = model.x[4]

if model.success and b_cost < 0 and b_time < 0:
    vtts = b_time / b_cost
    print(f"Value of Travel Time Saving: £{vtts:.2f}/hour")
else:
    print("Review estimation results before interpreting VTTS.")


Value of Travel Time Saving: £30.99/hour


In [9]:

# Observed mode shares
counts = np.bincount(y, minlength=4)
shares = counts / counts.sum()

# Null model: alternative-specific constants only
LL_null = np.sum(
    counts * np.log(shares)
)

# Our estimated MNL
LL_model = -model.fun

# McFadden's pseudo R-squared
rho_squared = 1 - (LL_model / LL_null)

print(f"Null Log-Likelihood: {LL_null:.2f}")
print(f"Model Log-Likelihood: {LL_model:.2f}")
print(f"McFadden Pseudo R²: {rho_squared:.4f}")


Null Log-Likelihood: -92322.86
Model Log-Likelihood: -74975.98
McFadden Pseudo R²: 0.1879


In [10]:

from scipy.optimize._numdiff import approx_derivative
from scipy.stats import norm

# Numerical Hessian of the negative log-likelihood
def gradient(params):
    return approx_derivative(
        negative_log_likelihood,
        params,
        args=(T, C, y),
        method="3-point"
    ).ravel()

hessian = approx_derivative(
    gradient,
    model.x,
    method="3-point"
)

# Conventional covariance matrix
cov_matrix = np.linalg.inv(hessian)

# Standard errors
standard_errors = np.sqrt(np.diag(cov_matrix))

# z-statistics and two-sided p-values
z_values = model.x / standard_errors
p_values = 2 * norm.sf(np.abs(z_values))

coefficient_table = pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": model.x,
    "Std. Error": standard_errors,
    "z-statistic": z_values,
    "p-value": p_values
})

display(coefficient_table.round(4))


,Parameter,Estimate,Std. Error,z-statistic,p-value
0,ASC_Cycle,-3.7981,0.0252,-150.5116,0.0
1,ASC_PT,-0.5037,0.0130,-38.6292,0.0
2,ASC_Car,-1.2234,0.0180,-67.9152,0.0
3,B_Time,-5.3102,0.0442,-120.0317,0.0
4,B_Cost,-0.1714,0.0034,-50.9459,0.0


In [11]:

# Person identifier
person_id = (
    df["household_id"].astype(str)
    + "_"
    + df["person_n"].astype(str)
)

# Design matrix: one row per trip and alternative
n = len(df)

X = np.zeros((n, 4, 5))

# Alternative-specific constants
X[:, 1, 0] = 1  # Cycle
X[:, 2, 1] = 1  # PT
X[:, 3, 2] = 1  # Car

# Time and cost variables
X[:, :, 3] = T
X[:, :, 4] = C

# Fitted utilities and choice probabilities
V = np.einsum("nik,k->ni", X, model.x)

V_shift = V - V.max(axis=1, keepdims=True)
exp_v = np.exp(V_shift)
P = exp_v / exp_v.sum(axis=1, keepdims=True)

# Score contribution for each trip
observed_X = X[np.arange(n), y]
expected_X = np.einsum("ni,nik->nk", P, X)

scores = observed_X - expected_X

# Sum scores within each person
score_df = pd.DataFrame(
    scores,
    columns=parameter_names
)
score_df["person_id"] = person_id.to_numpy()

cluster_scores = (
    score_df.groupby("person_id")[parameter_names]
    .sum()
    .to_numpy()
)

# Cluster-robust sandwich covariance
bread = np.linalg.inv(hessian)
meat = cluster_scores.T @ cluster_scores

cov_cluster = bread @ meat @ bread

# Finite-sample correction
G = len(cluster_scores)
K = len(model.x)
N = len(df)

cov_cluster *= (
    (G / (G - 1)) *
    ((N - 1) / (N - K))
)

cluster_se = np.sqrt(np.diag(cov_cluster))
cluster_z = model.x / cluster_se

cluster_table = pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": model.x,
    "Standard SE": standard_errors,
    "Cluster SE": cluster_se,
    "Cluster z": cluster_z
})

display(cluster_table.round(4))


,Parameter,Estimate,Standard SE,Cluster SE,Cluster z
0,ASC_Cycle,-3.7981,0.0252,0.0382,-99.3742
1,ASC_PT,-0.5037,0.0130,0.0182,-27.7449
2,ASC_Car,-1.2234,0.0180,0.0273,-44.7671
3,B_Time,-5.3102,0.0442,0.0632,-84.0337
4,B_Cost,-0.1714,0.0034,0.0046,-37.2776


In [12]:

# Calculate fitted utilities
V = np.einsum("nik,k->ni", X, model.x)

# Calculate choice probabilities
V_shift = V - V.max(axis=1, keepdims=True)
exp_V = np.exp(V_shift)
P = exp_V / exp_V.sum(axis=1, keepdims=True)

mode_names = [
    "Walk",
    "Cycle",
    "Public Transport",
    "Car"
]

# Observed shares
observed_shares = (
    np.bincount(y, minlength=4) / len(y) * 100
)

# Predicted shares
predicted_shares = P.mean(axis=0) * 100

comparison = pd.DataFrame({
    "Mode": mode_names,
    "Observed (%)": observed_shares,
    "Predicted (%)": predicted_shares,
    "Difference (pp)": predicted_shares - observed_shares
})

display(comparison.round(3))


,Mode,Observed (%),Predicted (%),Difference (pp)
0,Walk,17.596,17.596,0.0
1,Cycle,2.966,2.966,0.0
2,Public Transport,35.277,35.277,0.0
3,Car,44.161,44.160,-0.0


In [13]:

from sklearn.model_selection import GroupShuffleSplit

# Unique person identifier
person_id = (
    df["household_id"].astype(str)
    + "_"
    + df["person_n"].astype(str)
)

# Split observations by person
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(T, y, groups=person_id)
)

# Separate training and test arrays
T_train, T_test = T[train_idx], T[test_idx]
C_train, C_test = C[train_idx], C[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Training trips:", len(train_idx))
print("Test trips:", len(test_idx))

# Confirm no person appears in both sets
train_persons = set(person_id.iloc[train_idx])
test_persons = set(person_id.iloc[test_idx])

print(
    "Overlapping persons:",
    len(train_persons.intersection(test_persons))
)


Training trips: 64982
Test trips: 16104
Overlapping persons: 0


In [14]:

# Re-estimate using training data only

train_model = minimize(
    fun=negative_log_likelihood,
    x0=np.zeros(5),
    args=(T_train, C_train, y_train),
    method="L-BFGS-B",
    options={"maxiter": 500}
)

print("Converged:", train_model.success)
print("Training Log-Likelihood:", -train_model.fun)

train_parameters = pd.DataFrame({
    "Parameter": parameter_names,
    "Full-data Estimate": model.x,
    "Training Estimate": train_model.x
})

display(train_parameters.round(4))


Converged: True
Training Log-Likelihood: -59830.8143141763


,Parameter,Full-data Estimate,Training Estimate
0,ASC_Cycle,-3.7981,-3.8276
1,ASC_PT,-0.5037,-0.5185
2,ASC_Car,-1.2234,-1.2408
3,B_Time,-5.3102,-5.3693
4,B_Cost,-0.1714,-0.1733


In [15]:

from scipy.special import softmax
from sklearn.metrics import accuracy_score, log_loss

def predict_mnl(params, T_data, C_data):
    asc_cycle, asc_pt, asc_car, b_time, b_cost = params

    ASC = np.array([0, asc_cycle, asc_pt, asc_car])

    V = ASC + b_time * T_data + b_cost * C_data

    return softmax(V, axis=1)

# Probabilities on unseen test trips
P_test = predict_mnl(
    train_model.x,
    T_test,
    C_test
)

# Most likely mode
y_pred = P_test.argmax(axis=1)

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
test_logloss = log_loss(
    y_test,
    P_test,
    labels=[0, 1, 2, 3]
)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Log Loss: {test_logloss:.4f}")

# Observed versus predicted mode shares
test_comparison = pd.DataFrame({
    "Mode": mode_names,
    "Observed (%)": (
        np.bincount(y_test, minlength=4)
        / len(y_test) * 100
    ),
    "Predicted (%)": P_test.mean(axis=0) * 100
})

display(test_comparison.round(3))


Test Accuracy: 0.5841
Test Log Loss: 0.9406


,Mode,Observed (%),Predicted (%)
0,Walk,17.449,17.725
1,Cycle,3.086,2.941
2,Public Transport,36.165,35.430
3,Car,43.300,43.903


In [16]:

# Baseline probabilities
P_baseline = predict_mnl(
    train_model.x,
    T_test,
    C_test
)

# Scenario: 10% increase in PT fares
C_scenario = C_test.copy()
C_scenario[:, 2] *= 1.10

# Probabilities after fare increase
P_scenario = predict_mnl(
    train_model.x,
    T_test,
    C_scenario
)

# Predicted mode shares
baseline_shares = P_baseline.mean(axis=0) * 100
scenario_shares = P_scenario.mean(axis=0) * 100

fare_impact = pd.DataFrame({
    "Mode": mode_names,
    "Baseline (%)": baseline_shares,
    "Fare +10% (%)": scenario_shares,
    "Change (pp)": scenario_shares - baseline_shares
})

display(fare_impact.round(4))


,Mode,Baseline (%),Fare +10% (%),Change (pp)
0,Walk,17.7254,17.8197,0.0943
1,Cycle,2.9413,2.9745,0.0332
2,Public Transport,35.4300,34.9688,-0.4612
3,Car,43.9033,44.2370,0.3337


In [17]:

# Relative change in PT mode share
relative_pt_change = (
    scenario_shares[2] - baseline_shares[2]
) / baseline_shares[2]

# Arc-style finite-change elasticity using
# the baseline as the reference
fare_elasticity = relative_pt_change / 0.10

print(
    f"PT fare elasticity (finite-change): "
    f"{fare_elasticity:.4f}"
)

print(
    f"Relative change in PT share: "
    f"{relative_pt_change * 100:.3f}%"
)


PT fare elasticity (finite-change): -0.1302
Relative change in PT share: -1.302%


In [18]:

# Scenario: 10% reduction in PT travel time
T_scenario = T_test.copy()

T_scenario[:, 2] *= 0.90

# Predict choice probabilities
P_time_scenario = predict_mnl(
    train_model.x,
    T_scenario,
    C_test
)

# Predicted mode shares
time_scenario_shares = (
    P_time_scenario.mean(axis=0) * 100
)

time_impact = pd.DataFrame({
    "Mode": mode_names,
    "Baseline (%)": baseline_shares,
    "PT Time -10% (%)": time_scenario_shares,
    "Change (pp)": (
        time_scenario_shares - baseline_shares
    )
})

display(time_impact.round(4))

# Finite-change elasticity
pt_time_elasticity = (
    (time_scenario_shares[2] - baseline_shares[2])
    / baseline_shares[2]
) / (-0.10)

print(
    f"PT travel time elasticity: "
    f"{pt_time_elasticity:.4f}"
)


,Mode,Baseline (%),PT Time -10% (%),Change (pp)
0,Walk,17.7254,17.0240,-0.7015
1,Cycle,2.9413,2.6690,-0.2723
2,Public Transport,35.4300,39.8448,4.4148
3,Car,43.9033,40.4623,-3.4410


PT travel time elasticity: -1.2461
